<a href="https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** **"The Anatomy of Growing Content" (page 6, CONFIRMED)**. Growing pages are shown to be 37.6% longer and 20% younger than declining pages, framed as evidence that expanding thin pages helps. My methodology question: the paper's own correlation appendix (page 25) shows content_age_days and word_count are correlated at -0.5 — one of the strongest relationships in the whole matrix. Since older pages already tend to be shorter, could the word-count gap between growing and declining content partly be restating the age gap, rather than an independent factor? I'd ask whether a version of this comparison exists that holds age roughly constant (e.g., compares word count within similar age bands) before recommending "expand thin pages" as a standalone lever — this doesn't overturn the finding, but it would clarify whether length or age is doing more of the work. This mirrors a check I ran on my own capstone data: correlated inputs can make two separate-looking patterns actually be one pattern wearing two names.

**Finding 2: "What Predicts Growth?" (page 29, exploratory appendix)** — logistic regression, 71% holdout accuracy. The methodology section (page 36) states this uses an "80/20 split" but doesn't specify whether it's grouped by brand or randomly split at the content-row level. My methodology question: with 57 brands in the dataset, could some brands' pages appear in both the train and test sets under a random split? In my own Week-5 model, switching from a random split to a client-grouped split changed my headline numbers meaningfully (I found the gap itself was informative) — so I'd ask whether this 71% was measured on brands the model had already partially seen, since that's the single most common way a holdout number ends up higher than it would be in true deployment against a brand-new client. Secondarily, the paper doesn't report the base rate of growing-vs-declining in this specific sample, so it's hard to know how much of the 71% is real separation versus the label's natural class balance — small, easy addition, not a flaw in the finding itself.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before (random split):** Precision@10/20/50 = 0.900/0.900/0.920, base rate 0.550.

**After (grouped by client):** Precision@10/20/50 = 0.800/0.650/0.740, base rate 0.585.

The gap is real and explainable: a random split lets rows from the same client land in both train and test. Since clients differ in their typical CTR, position, and volume patterns (confirmed back in ML-08, where the baseline rule's average score differed ~3x between train and test clients), a model can partly learn "what this specific client's pages tend to look like" rather than a pattern that generalizes to a brand-new client. The random split's numbers are optimistic by roughly 0.10–0.25 precision points depending on K — that's the cost of memorization, not real skill. The honest number for how this model would perform on a client it's never seen is the grouped-split result, not the random one.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess, duckdb, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

REPO_URL = "https://github.com/ArinzeIhematulam/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

# Rebuild the same base feature frame as Week 5
base = con.sql("""
    WITH filtered AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    h1 AS (
        SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions) AS impressions_h1, SUM(gsc_clicks) AS clicks_h1,
               AVG(gsc_avg_position) AS avg_position_h1,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_h1,
               STDDEV(gsc_avg_position) AS position_std_h1
        FROM filtered WHERE day_of_month <= 15 GROUP BY content_hash_id
    ),
    h2 AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_h2
        FROM filtered WHERE day_of_month > 15 GROUP BY content_hash_id
    )
    SELECT h1.*, h2.clicks_h2
    FROM h1 JOIN h2 USING (content_hash_id)
    WHERE h1.clicks_h1 > 0
""").df()

base["declining"] = (base["clicks_h2"] < base["clicks_h1"]).astype(int)
features_clean = ["impressions_h1", "avg_position_h1", "active_days_h1", "position_std_h1"]

def precision_at_k(y_true, scores, k):
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return y_true.iloc[order].mean()

# BEFORE: random split (ignores that content repeats within a client)
X, y = base[features_clean].fillna(0), base["declining"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)

rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

print("=== RANDOM SPLIT (before) ===")
for k in [10, 20, 50]:
    print(f"Precision@{k}: {precision_at_k(y_test_r.reset_index(drop=True), scores_random, k):.3f}")
print(f"Base rate: {y_test_r.mean():.3f}")

# AFTER: grouped split by client
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(base, groups=base["client_hash_id"]))
train_df, test_df = base.iloc[train_idx], base.iloc[test_idx]

X_train_g, y_train_g = train_df[features_clean].fillna(0), train_df["declining"]
X_test_g, y_test_g = test_df[features_clean].fillna(0), test_df["declining"]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

print("\n=== GROUPED SPLIT (after) ===")
for k in [10, 20, 50]:
    print(f"Precision@{k}: {precision_at_k(y_test_g.reset_index(drop=True), scores_grouped, k):.3f}")
print(f"Base rate: {y_test_g.mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== RANDOM SPLIT (before) ===
Precision@10: 1.000
Precision@20: 0.900
Precision@50: 0.880
Base rate: 0.552

=== GROUPED SPLIT (after) ===
Precision@10: 0.800
Precision@20: 0.650
Precision@50: 0.740
Base rate: 0.585


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Attack checklist against features_clean = [impressions_h1, avg_position_h1, active_days_h1, position_std_h1]:

**Timeline:** all four features are aggregated over March 1–15 only; the label (declining) compares against March 16–31 — strictly before, no overlap. ✅

**Label-derived/sibling columns:** an earlier version of this model included ctr_h1 and scored a suspicious 1.00 precision@10. Investigating showed this came from pages with impressions_h1 = 1, clicks_h1 = 1 — a mechanical 100% CTR from a single lucky click, not a real signal. ctr_h1 was removed; the four features above showed no similar red flag when tested individually. ✅ (caught, not present)

**Product flags:** no FlyRank composite scores, health scores, or optimization flags are used as inputs anywhere in this pipeline. ✅
Split grouped by repeating entity: yes — by client_hash_id, confirmed above. ✅

**Base rate printed next to every metric:** yes, both splits report it (0.550 random, 0.585 grouped). ✅

**Feature importance sanity-checked:** permutation importance (from Week 5) showed impressions_h1 dominating (0.023) with the others contributing far less — no single feature towers over the rest the way ctr_h1 did, which is the expected pattern for an honest result. ✅

**Metrics computed out-of-fold:** yes — all precision@K numbers above are computed on the held-out test split only, never on training rows. ✅

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (too bold)**: "Both learned models clearly beat the Week-4 baseline on this held-out client set."

**Rewritten (safe language):** "In this observed sample, both models showed higher Precision@K than the Week-4 rule-based baseline when evaluated on 9 clients not seen during training — a directional result consistent with the idea that scale-independent signals (position, active days) generalize across clients better than a raw-magnitude score. This is decision-support evidence for trying a learned model in this lane, not a guarantee it will outperform the baseline on every future client or time period."

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.